# Simple Classification Project — Breast Cancer Dataset

Run each cell from top to bottom. The goal is to classify the target as malignant or benign. This dataset is for machine-learning practice only, not medical diagnosis.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## 2. Load the dataset
Keep `breast_cancer_classification.csv` in the same folder.

In [ ]:
df = pd.read_csv("breast_cancer_classification.csv")
print("Rows and columns:", df.shape)
df.head()

## 3. Understand the data

In [ ]:
df.info()

In [ ]:
print("Missing values by column:")
print(df.isnull().sum())

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Class counts:")
print(df["target"].value_counts())

## 4. Data cleaning
Remove duplicate rows, remove records without a target label, and fill missing numeric feature values with the median. Imputation and scaling are also placed inside model pipelines to avoid learning preprocessing values from validation/test folds.

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["target"])

X = df.drop(columns=["target"])
y = df["target"]

print("Shape after removing duplicates/missing labels:", df.shape)
print("Target classes:", y.unique())

## 5. Split data into train and test sets
`stratify=y` keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 6. Create classification models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=42))
    ])
}

## 7. Compare models using cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1
    )
    results.append([name, scores.mean()])

cv_results = pd.DataFrame(results, columns=["Model", "Average CV Accuracy"])
cv_results.sort_values("Average CV Accuracy", ascending=False)

## 8. Tune the models using GridSearchCV

In [ ]:
parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1, 10]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    },
    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"]
    },
    "Decision Tree": {
        "model__max_depth": [None, 3, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10]
    }
}

In [ ]:
best_models = {}
tuning_results = []

for name in models:
    search = GridSearchCV(
        models[name],
        parameter_grids[name],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    tuning_results.append([name, search.best_score_, search.best_params_])

pd.DataFrame(
    tuning_results,
    columns=["Model", "Best CV Accuracy", "Best Parameters"]
).sort_values("Best CV Accuracy", ascending=False)

## 9. Evaluate all tuned models on test data

In [ ]:
test_results = []

for name, model in best_models.items():
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, pos_label="malignant", zero_division=0)
    recall = recall_score(y_test, predictions, pos_label="malignant", zero_division=0)
    f1 = f1_score(y_test, predictions, pos_label="malignant", zero_division=0)
    test_results.append([name, accuracy, precision, recall, f1])

test_results = pd.DataFrame(
    test_results,
    columns=["Model", "Accuracy", "Malignant Precision", "Malignant Recall", "Malignant F1"]
).sort_values("Accuracy", ascending=False)

test_results

In [ ]:
best_name = test_results.iloc[0]["Model"]
print("Highest test accuracy model:", best_name)

## 10. Confusion matrix and classification report

In [ ]:
best_model = best_models[best_name]
predictions = best_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, labels=["malignant", "benign"], values_format="d"
)
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, predictions, zero_division=0))

## 11. Conclusion
Use the actual values in `test_results`. Compare accuracy, precision, recall, and F1. Recall for malignant cases tells how many actual malignant cases were detected. This educational model must not be used for medical diagnosis.